In [3]:
# !pip install optuna

In [4]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [6]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [8]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2025-06-12 19:42:50,424] A new study created in memory with name: no-name-c1941bc7-6f2e-4d73-a566-82e6fa02c181
[I 2025-06-12 19:42:51,398] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 112, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-06-12 19:42:52,696] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 153, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-06-12 19:42:53,735] Trial 2 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 115, 'max_depth': 10}. Best is trial 2 with value: 0.7709497206703911.
[I 2025-06-12 19:42:55,246] Trial 3 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 176, 'max_depth': 6}. Best is trial 2 with value: 0.7709497206703911.
[I 2025-06-12 19:42:56,519] Trial 4 finished with value: 0.7541899441340782 and parameters: {'n_estimators': 158, 'max_depth': 3}. Best is trial 2 with value: 0.77094972

In [9]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 118, 'max_depth': 15}


In [10]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


## Sampler in optuna 


from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score




In [11]:
# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize

In [12]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2025-06-12 19:43:53,492] A new study created in memory with name: no-name-7e4b8fe9-c429-44cb-ac26-eea953a10aee
[I 2025-06-12 19:43:55,462] Trial 0 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 190, 'max_depth': 15}. Best is trial 0 with value: 0.7709497206703911.
[I 2025-06-12 19:43:56,243] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 75, 'max_depth': 14}. Best is trial 0 with value: 0.7709497206703911.
[I 2025-06-12 19:43:58,256] Trial 2 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 179, 'max_depth': 9}. Best is trial 0 with value: 0.7709497206703911.
[I 2025-06-12 19:43:58,884] Trial 3 finished with value: 0.7523277467411545 and parameters: {'n_estimators': 59, 'max_depth': 3}. Best is trial 0 with value: 0.7709497206703911.
[I 2025-06-12 19:43:59,737] Trial 4 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 59, 'max_depth': 15}. Best is trial 0 with value: 0.770949720

In [13]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7821229050279329
Best hyperparameters: {'n_estimators': 119, 'max_depth': 20}


In [14]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


In [15]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [16]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2025-06-12 19:44:54,399] A new study created in memory with name: no-name-f515abf1-5662-43df-bc34-cbb55ec4ded9
[I 2025-06-12 19:44:55,218] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-06-12 19:44:56,608] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-06-12 19:44:57,085] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-06-12 19:44:58,010] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-06-12 19:44:58,933] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [17]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [18]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74


## Optuna Visualizations

In [19]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [20]:
# 1. Optimization History
plot_optimization_history(study).show()

In [21]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [22]:
# 3. Slice Plot
plot_slice(study).show()

In [23]:
# 4. Contour Plot
plot_contour(study).show()

In [24]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [25]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [26]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [27]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2025-06-12 19:45:17,282] A new study created in memory with name: no-name-a547ce84-5237-428c-bb26-59c2b28454b9
[I 2025-06-12 19:45:17,366] Trial 0 finished with value: 0.7150837988826816 and parameters: {'classifier': 'SVM', 'C': 79.43014362006525, 'kernel': 'rbf', 'gamma': 'scale'}. Best is trial 0 with value: 0.7150837988826816.
[I 2025-06-12 19:45:18,213] Trial 1 finished with value: 0.7635009310986964 and parameters: {'classifier': 'RandomForest', 'n_estimators': 96, 'max_depth': 18, 'min_samples_split': 8, 'min_samples_leaf': 4, 'bootstrap': True}. Best is trial 1 with value: 0.7635009310986964.
[I 2025-06-12 19:45:20,416] Trial 2 finished with value: 0.7560521415270017 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 226, 'learning_rate': 0.06246844123232072, 'max_depth': 4, 'min_samples_split': 10, 'min_samples_leaf': 8}. Best is trial 1 with value: 0.7635009310986964.
[I 2025-06-12 19:45:22,253] Trial 3 finished with value: 0.7430167597765363 and parameters

In [28]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.1296449261075851, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [29]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.715084,2025-06-12 19:45:17.285302,2025-06-12 19:45:17.366159,0 days 00:00:00.080857,79.430144,NaN,SVM,scale,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.763501,2025-06-12 19:45:17.368846,2025-06-12 19:45:18.213078,0 days 00:00:00.844232,NaN,True,RandomForest,NaN,NaN,NaN,18.0,4.0,8.0,96.0,COMPLETE
2,2,0.756052,2025-06-12 19:45:18.215614,2025-06-12 19:45:20.415257,0 days 00:00:02.199643,NaN,NaN,GradientBoosting,NaN,NaN,0.062468,4.0,8.0,10.0,226.0,COMPLETE
3,3,0.743017,2025-06-12 19:45:20.417607,2025-06-12 19:45:22.252498,0 days 00:00:01.834891,NaN,NaN,GradientBoosting,NaN,NaN,0.163997,17.0,9.0,10.0,92.0,COMPLETE
4,4,0.774674,2025-06-12 19:45:22.254027,2025-06-12 19:45:22.309920,0 days 00:00:00.055893,0.189386,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.789572,2025-06-12 19:46:18.117761,2025-06-12 19:46:18.171919,0 days 00:00:00.054158,0.133281,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.770950,2025-06-12 19:46:18.174036,2025-06-12 19:46:18.237622,0 days 00:00:00.063586,0.170020,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.715084,2025-06-12 19:46:18.239616,2025-06-12 19:46:18.293858,0 days 00:00:00.054242,0.200777,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.789572,2025-06-12 19:46:18.294871,2025-06-12 19:46:18.345865,0 days 00:00:00.050994,0.130062,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [30]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 78
RandomForest        12
GradientBoosting    10
Name: count, dtype: int64

In [31]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

params_classifier
GradientBoosting    0.747300
RandomForest        0.765518
SVM                 0.776942
Name: value, dtype: float64

In [32]:
# 1. Optimization History
plot_optimization_history(study).show()

In [33]:
# 3. Slice Plot
plot_slice(study).show()

In [34]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()    

In [37]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")


[I 2025-06-12 19:47:37,706] A new study created in memory with name: no-name-a54337c3-f2db-41bd-a970-a65e710ff272


[0]	train-mlogloss:0.98219	eval-mlogloss:0.98365
[1]	train-mlogloss:0.89456	eval-mlogloss:0.88559
[2]	train-mlogloss:0.80735	eval-mlogloss:0.79107
[3]	train-mlogloss:0.72517	eval-mlogloss:0.70520
[4]	train-mlogloss:0.66029	eval-mlogloss:0.63920
[5]	train-mlogloss:0.59872	eval-mlogloss:0.57221
[6]	train-mlogloss:0.54574	eval-mlogloss:0.51337
[7]	train-mlogloss:0.50490	eval-mlogloss:0.47294
[8]	train-mlogloss:0.46813	eval-mlogloss:0.43165
[9]	train-mlogloss:0.43003	eval-mlogloss:0.39228
[10]	train-mlogloss:0.39698	eval-mlogloss:0.35804
[11]	train-mlogloss:0.36852	eval-mlogloss:0.32748
[12]	train-mlogloss:0.34442	eval-mlogloss:0.30162
[13]	train-mlogloss:0.32318	eval-mlogloss:0.27785
[14]	train-mlogloss:0.30742	eval-mlogloss:0.26131
[15]	train-mlogloss:0.28945	eval-mlogloss:0.24116
[16]	train-mlogloss:0.27587	eval-mlogloss:0.22584
[17]	train-mlogloss:0.26408	eval-mlogloss:0.21296
[18]	train-mlogloss:0.25272	eval-mlogloss:0.20033
[19]	train-mlogloss:0.24479	eval-mlogloss:0.19096
[20]	train

[I 2025-06-12 19:47:39,560] Trial 0 finished with value: 1.0 and parameters: {'lambda': 0.0053756740193569574, 'alpha': 2.3961480124483085e-08, 'eta': 0.09531920517143157, 'gamma': 0.11584042713761838, 'max_depth': 9, 'min_child_weight': 5, 'subsample': 0.4928767937806364, 'colsample_bytree': 0.9666856335903532}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:0.80387	eval-mlogloss:0.80711
[1]	train-mlogloss:0.64810	eval-mlogloss:0.62496
[2]	train-mlogloss:0.50925	eval-mlogloss:0.47088
[3]	train-mlogloss:0.40284	eval-mlogloss:0.35442
[4]	train-mlogloss:0.35294	eval-mlogloss:0.30261
[5]	train-mlogloss:0.30908	eval-mlogloss:0.25246
[6]	train-mlogloss:0.27700	eval-mlogloss:0.21415
[7]	train-mlogloss:0.26666	eval-mlogloss:0.20378
[8]	train-mlogloss:0.26668	eval-mlogloss:0.20491
[9]	train-mlogloss:0.26568	eval-mlogloss:0.20068
[10]	train-mlogloss:0.25431	eval-mlogloss:0.18910
[11]	train-mlogloss:0.25415	eval-mlogloss:0.19029
[12]	train-mlogloss:0.25244	eval-mlogloss:0.18866
[13]	train-mlogloss:0.25164	eval-mlogloss:0.18820
[14]	train-mlogloss:0.25105	eval-mlogloss:0.18699
[15]	train-mlogloss:0.25105	eval-mlogloss:0.18665
[16]	train-mlogloss:0.25026	eval-mlogloss:0.18585
[17]	train-mlogloss:0.24996	eval-mlogloss:0.18570
[18]	train-mlogloss:0.25032	eval-mlogloss:0.18638
[19]	train-mlogloss:0.24987	eval-mlogloss:0.18615
[20]	train

[I 2025-06-12 19:47:40,524] Trial 1 finished with value: 1.0 and parameters: {'lambda': 0.002479850613871405, 'alpha': 2.3506377332967465e-05, 'eta': 0.2624896817846184, 'gamma': 0.016050865779911612, 'max_depth': 5, 'min_child_weight': 7, 'subsample': 0.5437822011629895, 'colsample_bytree': 0.9067501840159709}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:0.97446	eval-mlogloss:0.96995
[1]	train-mlogloss:0.92311	eval-mlogloss:0.90863
[2]	train-mlogloss:0.83775	eval-mlogloss:0.81549


[I 2025-06-12 19:48:02,806] Trial 48 pruned. Trial was pruned at iteration 2.


[0]	train-mlogloss:1.01814	eval-mlogloss:1.01905
[1]	train-mlogloss:0.95880	eval-mlogloss:0.95049


[I 2025-06-12 19:48:02,968] Trial 49 pruned. Trial was pruned at iteration 2.


Best trial: {'lambda': 0.0053756740193569574, 'alpha': 2.3961480124483085e-08, 'eta': 0.09531920517143157, 'gamma': 0.11584042713761838, 'max_depth': 9, 'min_child_weight': 5, 'subsample': 0.4928767937806364, 'colsample_bytree': 0.9666856335903532}
Best accuracy: 1.0


In [36]:
! pip install optuna-integration[xgboost]

In [38]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()